In [14]:
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from sklearn.model_selection import TimeSeriesSplit
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.neighbors import KNeighborsRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, mean_absolute_percentage_error


*Voy a usar datos del bitcoin solo para predecir*

In [15]:
# carga y preparación de datos
btc = pd.read_csv("../data/full_data.csv")
btc['Date'] = pd.to_datetime(btc['Date'])
btc.set_index('Date', inplace=True)

# crear targets futuros (Close_t+1 ... Close_t+7)
for i in range(1,8):
    btc[f'Close_t+{i}'] = btc['Close'].shift(-i)
btc.dropna(inplace=True)

# features
features = [
    'Open','High','Low','Volume','Volatility','SMA_7','SMA_30'
]
targets = [f'Close_t+{i}' for i in range(1,8)]

X = btc[features].copy()
Y = btc[targets].copy()

print("Valores nulos en X:", X.isnull().sum().sum())
print("Valores nulos en Y:", Y.isnull().sum().sum())

# escalado
mapper = ColumnTransformer(
    transformers=[('scaler', StandardScaler(), features)],
    remainder='drop'
)

# valid. temporal
n_splits = 5
tvt = TimeSeriesSplit(n_splits=n_splits)

np.random.seed(42)
random_state = 42

# modelos base
models = {
    'LinearRegression': LinearRegression(),
    'RandomForest': RandomForestRegressor(
        n_estimators=100,
        random_state=random_state
    ),
    'KNN': KNeighborsRegressor(n_neighbors=21)
}


Valores nulos en X: 0
Valores nulos en Y: 0


In [16]:
# funcion para entrenar y evaluar un modelo
from sklearn.multioutput import MultiOutputRegressor
def run_model(model_name, model):
    print(f"Ejecutando modelo: {model_name}")
    
    fold_results = []
    predictions_results = []

    for fold, (train_val_idx, test_idx) in enumerate(tvt.split(X)):
        X_train_val, X_test = X.iloc[train_val_idx], X.iloc[test_idx]
        Y_train_val, Y_test = Y.iloc[train_val_idx], Y.iloc[test_idx]

        # divido en train y val (80/20)
        val_size = int(len(X_train_val) * 0.2)
        X_train, X_val = X_train_val.iloc[:-val_size], X_train_val.iloc[-val_size:]
        Y_train, Y_val = Y_train_val.iloc[:-val_size], Y_train_val.iloc[-val_size:]

        # verifico si el modelo soporta multiples salidas
        model_to_use = model
        try:
            # prueba de soporte multisalida
            model.fit(X_train[:5], Y_train[:5])
        except ValueError:
            # si no soporta, envolverlo
            model_to_use = MultiOutputRegressor(model)
        
        # creo pipeline
        pipeline = Pipeline([('mapper', mapper), ('model', model_to_use)])
        
        # entreno
        pipeline.fit(X_train, Y_train)

        # predecir
        Y_val_pred = pipeline.predict(X_val)
        Y_val_pred_df = pd.DataFrame(Y_val_pred, columns=Y_val.columns, index=X_val.index)

        # metricas
        val_MAE = mean_absolute_error(Y_val, Y_val_pred_df)
        val_MSE = mean_squared_error(Y_val, Y_val_pred_df)
        val_RMSE = np.sqrt(val_MSE)
        val_MAPE = mean_absolute_percentage_error(Y_val, Y_val_pred_df) * 100

        fold_results.append({
            'fold': fold + 1,
            'val_mae': val_MAE,
            'val_mse': val_MSE,
            'val_rmse': val_RMSE,
            'val_mape': val_MAPE,
            'train_start': X_train.index.min(),
            'train_end': X_train.index.max(),
            'val_start': X_val.index.min(),
            'val_end': X_val.index.max(),
        })

        # guardo predicciones
        comparison = pd.concat([
            Y_val.add_suffix('_real'),
            Y_val_pred_df.add_suffix('_pred')
        ], axis=1)
        comparison['fold'] = fold + 1
        predictions_results.append(comparison)

    results_df = pd.DataFrame(fold_results)
    predictions_df = pd.concat(predictions_results, ignore_index=False)

    print(f"Modelo {model_name} completado. Promedio MAE: {results_df['val_mae'].mean():.4f}, RMSE: {results_df['val_rmse'].mean():.4f}")
    
    return results_df, predictions_df

In [17]:
# ejecutar todos los modelos
all_results = {}
all_predictions = {}

for model_name, model in models.items():
    try: 
        print(f"\n{'-'*50}")
        res, pred = run_model(model_name, model)
        all_results[model_name] = res
        all_predictions[model_name] = pred
        print(f"{model_name} ejecutado correctamente.")
    except Exception as e:
        print(f"Error al ejecutar {model_name}: {e}")

# resumen final
print("\nResumen general de resultados:")
for model_name, res in all_results.items():
    mean_mae = res['val_mae'].mean()
    mean_rmse = res['val_rmse'].mean()
    print(f" - {model_name}: MAE promedio = {mean_mae:.4f}, RMSE promedio = {mean_rmse:.4f}")


--------------------------------------------------
Ejecutando modelo: LinearRegression
Modelo LinearRegression completado. Promedio MAE: 1354.5606, RMSE: 1904.8429
LinearRegression ejecutado correctamente.

--------------------------------------------------
Ejecutando modelo: RandomForest
Modelo RandomForest completado. Promedio MAE: 3747.9539, RMSE: 5744.9147
RandomForest ejecutado correctamente.

--------------------------------------------------
Ejecutando modelo: KNN
Modelo KNN completado. Promedio MAE: 3552.8680, RMSE: 5380.0616
KNN ejecutado correctamente.

Resumen general de resultados:
 - LinearRegression: MAE promedio = 1354.5606, RMSE promedio = 1904.8429
 - RandomForest: MAE promedio = 3747.9539, RMSE promedio = 5744.9147
 - KNN: MAE promedio = 3552.8680, RMSE promedio = 5380.0616


In [18]:
# grafico de métricas por fold (por modelo)
for model_name, res in all_results.items():
    if res.empty:
        print(f"No hay resultados para {model_name}, se omite el gráfico.")
        continue

    metrics = ["val_mae", "val_mse", "val_rmse", "val_mape"]
    metric_titles = {
        "val_mae": "Error Absoluto Medio (MAE)",
        "val_mse": "Error Cuadrático Medio (MSE)",
        "val_rmse": "Raíz del Error Cuadrático Medio (RMSE)",
        "val_mape": "Error Porcentual Medio (MAPE %)"
    }

    fig = make_subplots(rows=2, cols=2, subplot_titles=[metric_titles[m] for m in metrics])

    for i, m in enumerate(metrics):
        row = i // 2 + 1
        col = i % 2 + 1
        fig.add_trace(
            go.Scatter(
                x=res["fold"],
                y=res[m],
                mode="lines+markers",
                name=m.upper(),
                line=dict(width=2),
                marker=dict(size=6)
            ),
            row=row, col=col
        )

    fig.update_layout(
        title=f"Métricas por fold - {model_name}",
        height=650,
        width=950,
        showlegend=False,
        template="plotly_white",
        font=dict(size=13),
        margin=dict(t=80, b=50, l=50, r=50)
    )

    fig.update_xaxes(title_text="Fold")
    fig.update_yaxes(title_text="Valor de la métrica")

    fig.show()
	

In [19]:
# grafico predicciones vs real (t+1 a t+7)
from plotly.colors import qualitative

for model_name, pred in all_predictions.items():
    if pred.empty:
        print(f"No hay predicciones para {model_name}, se omite el gráfico.")
        continue

    fig = go.Figure()
    days = [f't+{i}' for i in range(8)]
    color_palette = qualitative.Plotly  # paleta estándar (10 colores distintos)

    for i, day in enumerate(days, start=1):
        real_col = f'Close_{day}_real' if f'Close_{day}_real' in pred.columns else f'Close_t+{i}_real'
        pred_col = f'Close_{day}_pred' if f'Close_{day}_pred' in pred.columns else f'Close_t+{i}_pred'

        if real_col not in pred.columns or pred_col not in pred.columns:
            print(f"Columnas para {day} no encontradas en {model_name}, se omite.")
            continue

        color = color_palette[(i - 1) % len(color_palette)]

        # linea real
        fig.add_trace(go.Scatter(
            x=pred.index,
            y=pred[real_col],
            mode='lines',
            name=f'Real {day}',
            line=dict(color=color, width=2)
        ))

        # linea predicha
        fig.add_trace(go.Scatter(
            x=pred.index,
            y=pred[pred_col],
            mode='lines',
            name=f'Pred {day}',
            line=dict(color="red", width=2, dash='dash')
        ))

    fig.update_layout(
        title=f'Predicciones vs Real - {model_name}',
        xaxis_title='Fecha',
        yaxis_title='Precio BTC (USD)',
        width=1100,
        height=600,
        template='plotly_white',
        font=dict(size=13),
        legend=dict(
            orientation="h",
            yanchor="bottom",
            y=-0.3,
            xanchor="center",
            x=0.5
        ),
        margin=dict(t=80, b=80)
    )

    fig.show()
print("Fecha mínima:", pred.index.min())
print("Fecha máxima:", pred.index.max())

Fecha mínima: 2019-04-11 00:00:00
Fecha máxima: 2025-01-20 00:00:00


In [20]:
# tabla resumen de errores por modelo
summary = []

for model_name, res in all_results.items():
    # Validar que el DataFrame no esté vacío o sea None
    if res is None or res.empty:
        print(f"El modelo '{model_name}' no tiene resultados, se omite del resumen.")
        continue

    # calculo promedio de métricas
    avg_mae = res['val_mae'].mean()
    avg_rmse = res['val_rmse'].mean()
    avg_mape = res['val_mape'].mean()  # ya está en %

    summary.append({
        'Modelo': model_name,
        'MAE': round(avg_mae, 4),
        'RMSE': round(avg_rmse, 4),
        'MAPE (%)': round(avg_mape, 2)
    })

# convertir a DataFrame y ordenar
summary_df = (
    pd.DataFrame(summary)
    .sort_values(by='MAPE (%)', ascending=True)
    .reset_index(drop=True)
)

# mostrar tabla
print("\nResumen de métricas promedio por modelo:\n")

try:
    from IPython.display import display
    display(
        summary_df.style.set_table_styles([
            {'selector': 'th', 'props': [('background-color', '#003366'), ('color', 'white'), ('font-weight', 'bold')]},
            {'selector': 'td', 'props': [('text-align', 'center')]}
        ]).set_properties(**{'border-color': 'black'})
    )
except Exception:
    print(summary_df.to_string(index=False))


Resumen de métricas promedio por modelo:



,Modelo,MAE,RMSE,MAPE (%)
0,LinearRegression,1354.560600,1904.842900,4.770000
1,RandomForest,3747.953900,5744.914700,10.840000
2,KNN,3552.868000,5380.061600,11.240000


Calcula el promedio de cada métrica a lo largo de todos los folds:

#MAE (Mean Absolute Error): error promedio absoluto entre el valor real y el predicho.
→ mide en unidades del precio (por ejemplo, USD).
→ cuanto más bajo, mejor.

#RMSE (Root Mean Square Error): error cuadrático medio. Penaliza errores grandes.
→ también en USD, y debe ser lo más bajo posible.

#MAPE (Mean Absolute Percentage Error): error relativo expresado en porcentaje.
→ muestra cuánto se desvió la predicción en promedio con respecto al valor real (%).
→ un MAPE del 10 % significa que el modelo, en promedio, se equivoca un 10 % del valor real.

In [21]:
#gráfico comparativo de métricas
fig = make_subplots(rows=1, cols=3, subplot_titles=("MAE", "RMSE", "MAPE (%)"))

fig.add_trace(go.Bar(x=summary_df["Modelo"], y=summary_df["MAE"], name="MAE"), row=1, col=1)
fig.add_trace(go.Bar(x=summary_df["Modelo"], y=summary_df["RMSE"], name="RMSE"), row=1, col=2)
fig.add_trace(go.Bar(x=summary_df["Modelo"], y=summary_df["MAPE (%)"], name="MAPE (%)"), row=1, col=3)

fig.update_layout(title="Comparación de métricas por modelo", height=500, width=1000)
fig.show()